# QP 2 Analysis Playground

In [2]:
library(tidyverse)
library(ggplot2)
library(easystats)
library(data.table)
library(dtplyr)
library(arrow)


── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.1.4     ✔ readr     2.1.5
✔ forcats   1.0.0     ✔ stringr   1.5.1
✔ ggplot2   4.0.1     ✔ tibble    3.2.1
✔ lubridate 1.9.4     ✔ tidyr     1.3.1
✔ purrr     1.0.4     
── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors
# Attaching packages: easystats 0.7.5
✔ bayestestR  0.17.0   ✔ correlation 0.8.8 
✔ datawizard  1.3.0    ✔ effectsize  1.0.1 
✔ insight     1.4.4    ✔ modelbased  0.13.1
✔ performance 0.15.3   ✔ parameters  0.28.3
✔ report      0.6.2    ✔ see         0.12.0



Attaching package: 'data.table'


The following objects are masked from 'package:lubridate':

    hour, isoweek, mday, minute, month, quarter, second, wday, week,
    yday, year


The following objects are masked fr

In [ ]:
# 1. Parse set of CSV output files into single DF, write directly to RDS


# files <- list.files(path = "D:/BNC Full Data/12-9_5PM Run/CSV",
#                     pattern = "\\.csv$",
#                     full.names = TRUE)

# df_full <- read_csv(files, id = "file_name")

# write_rds(df_full, "D:/BNC Full Data/12-9_5PM Run/12-9_5PM_Full-Data_UNPROCESSED.rds")


In [3]:
# 2. Read directly from unprocessed RDS. 

df_full <- readRDS("D:/BNC Full Data/12-9_5PM Run/12-9_5PM_Full-Data_UNPROCESSED.rds")

In [ ]:
# # 3. Do some processing to tag index of first NP, sort some tihngs, set baseline for factors, etc. 
# head(df_full)

# Remove duplicates

# df_processed <- df_full %>% 
#     arrange(Sentence_ID) %>% # Removes duplicates
#         group_by(Sentence_Text) %>%
#         mutate(first_Sentence_ID = first(Sentence_ID)) %>%
#         filter(Sentence_ID == first_Sentence_ID) %>%
#         ungroup() %>%
#         select(-first_Sentence_ID)


df_processed <- df_processed %>%
    arrange(Sentence_ID, Word_Token_Index) %>% 
    group_by(Sentence_ID) %>% 
    mutate(
        prev_is_NP = lag(Is_NP, default = FALSE),
        prev_NP_Head_Text = lag(NP_Head_Text),

        first_token_of_NP = Is_NP & (!prev_is_NP | NP_Head_Text != prev_NP_Head_Text)
    ) %>% 
    ungroup() %>% 
    select(-prev_is_NP, -prev_NP_Head_Text) %>% 
#Propogates index of first NP down to the full phrase
    group_by(Sentence_ID, np_id = consecutive_id(Phrase_Token)) %>% 
    mutate(
        np_start_idx = ifelse(
            is.na (Phrase_Token) | Is_NP == FALSE, 
            NA,
            min(Word_Token_Index)
        )
    ) %>% 
    ungroup() %>% 
    select(-np_id) %>% 
# Creates within_file IDS and Within Chunk Ids
    mutate(
        within_file_id = str_extract(Sentence_ID, "(?<=_)\\d+") %>% 
        as.integer
    ) %>% 
    mutate(within_chunk_id = ((within_file_id %% 550) + 1)) %>% 
#Renames and sets baseline vals for analysis
    mutate(
        definiteness = factor(NP_Definiteness,
        levels = c("indefinite", "definite"),
        labels = c("indef", "def"))
    ) %>% 
    mutate(
        argPos = factor(
            NP_Argument,
            levels = c("dir_object", "subject"),
            labels = c("obj", "sbj")
        )
    ) %>% 
    mutate(surprisal = Phrase_Surprisal)


write_rds(df_processed, "D:/BNC Full Data/12-9_5PM Run/12-9_5PM_Full-Data.rds")
write_parquet(df_processed, "Results 12-9 5PM/12-9_5PM_Full-Data.parquet")



In [3]:
# Read directly from parquet file

ds_full <- open_dataset("Results 12-9 5PM/12-9_5PM_Full-Data.parquet")

print(nrow(ds_full))



[1] 103426691


In [ ]:
df_full <- ds_full %>% collect()
summary(df_full)


  file_name         Sentence_ID          Filename           Modality        
 Length:103426691   Length:103426691   Length:103426691   Length:103426691  
 Class :character   Class :character   Class :character   Class :character  
 Mode  :character   Mode  :character   Mode  :character   Mode  :character  
                                                                            
                                                                            
                                                                            
                                                                            
 Sentence_Text      Sent_Verb_Count  Sent_Auxiliary_Count Sent_Subject_Count
 Length:103426691   Min.   : 0.000   Min.   : 0.00        Min.   : 0.000    
 Class :character   1st Qu.: 2.000   1st Qu.: 1.00        1st Qu.: 1.000    
 Mode  :character   Median : 3.000   Median : 1.00        Median : 2.000    
                    Mean   : 3.072   Mean   : 1.66        Mean   : 1.919    

In [9]:
# Filter based on filtering criteria

df_filtered <- df_full %>% 

    filter(
        Is_NP == TRUE, # Filtering Criteria
        Is_Head_Noun == TRUE,
        Modality == "written", 
        Sent_Verb_Count == 1,
        Sent_Auxiliary_Count == 0,
        Sent_Subject_Count == 1,
        Sent_Tot_Obj_Count %in% 1,
        Sent_Dir_Object_Count == 1 ,
        Sent_Ind_Object_Count == 0,
        Sent_Sub_Conj_Count == 0,
        Sent_Coord_Conj_Count == 0, 
        Clausal_Complement_Count == 0,
        Sent_Relative_Clause_Count == 0, 
        Sent_Adv_Clause_Count == 0, 
        Sent_Prep_Phrase_Count == 0,
        Sent_Comma_Count == 0,
        !str_detect(Sentence_Text, "\\?"),
        definiteness  %in% c("def", "indef"),
        argPos %in% c("sbj", "obj"),
        Sent_Transitive == TRUE,
        !is.na(surprisal)
    ) %>% 
    group_by(Sentence_ID) %>% 
    filter(n() == 2 & n_distinct(argPos) == 2) %>% 
    ungroup() %>%
    select(Sentence_ID, Sentence_Text, surprisal, argPos, definiteness, np_start_idx, within_file_id, within_chunk_id) 

write_parquet(df_filtered, "Results 12-9 5PM/12-9_5PM_FILTERED-Data.parquet")
write_rds(df_filtered, "Results 12-9 5PM/12-9_5PM_FILTERED-Data.rds")

In [10]:
summary(df_filtered)

 Sentence_ID        Sentence_Text        surprisal      argPos     definiteness
 Length:3386        Length:3386        Min.   : 2.264   obj:1693   indef:1282  
 Class :character   Class :character   1st Qu.:13.492   sbj:1693   def  :2104  
 Mode  :character   Mode  :character   Median :15.781                          
                                       Mean   :16.156                          
                                       3rd Qu.:18.609                          
                                       Max.   :35.219                          
  np_start_idx    within_file_id  within_chunk_id
 Min.   : 0.000   Min.   :  2.0   Min.   :  2.0  
 1st Qu.: 0.000   1st Qu.:123.0   1st Qu.: 81.0  
 Median : 2.000   Median :250.0   Median :205.0  
 Mean   : 1.664   Mean   :262.5   Mean   :218.8  
 3rd Qu.: 3.000   3rd Qu.:396.0   3rd Qu.:338.0  
 Max.   :16.000   Max.   :568.0   Max.   :512.0  

In [4]:
# Rm duplicate sentences
# whitelist <- ds_full %>% 
#     select(Sentence_Text, Sentence_ID) %>% 
#     distinct() %>% 
#     group_by(Sentence_Text) %>% 
#     summarise(kept_id = min(Sentence_ID, na.rm = TRUE)) %>% 
#     ungroup()

# ds_filtered <- ds_full %>% 
# semi_join(whitelist, by = c("Sentence_ID" = "kept_id"))

ds_filtered <- ds_full %>% 
    group_by(Sentence_Text) %>% 
    mutate(first_id = min(Sentence_ID, na.rm = TRUE)) %>% 
    filter(Sentence_ID == first_id) %>% 
    ungroup() %>% 
    select(-first_id)

ds_fil_count <- ds_filtered %>% count() %>% collect()
print(ds_fil_count)



# A tibble: 1 × 1
          n
      <int>
1 103426691


In [ ]:
df_filtered <- df_full %>% 
    arrange(Sentence_ID) %>% # Removes duplicates
        group_by(Sentence_Text) %>%
        mutate(first_Sentence_ID = first(Sentence_ID)) %>%
        filter(Sentence_ID == first_Sentence_ID) %>%
        ungroup() %>%
        select(-first_Sentence_ID) %>% 
    filter(Is_NP == TRUE, # Filtering Criteria
            Is_Head_Noun == TRUE,
            Modality == "written", 
            Sent_Verb_Count == 1,
            Sent_Auxiliary_Count == 0,
            Sent_Subject_Count == 1,
            Sent_Tot_Obj_Count %in% 1,
            Sent_Dir_Object_Count == 1 ,
            Sent_Ind_Object_Count == 0,
            Sent_Sub_Conj_Count == 0,
            Sent_Coord_Conj_Count == 0, 
            Clausal_Complement_Count == 0,
            Sent_Relative_Clause_Count == 0, 
            Sent_Adv_Clause_Count == 0, 
            Sent_Prep_Phrase_Count == 0,
            Sent_Comma_Count == 0,
            !str_detect(Sentence_Text, "\\?"),
            NP_Definiteness  %in% c("definite", "indefinite"),
            NP_Argument %in% c("subject", "dir_object"),
            Sent_Transitive == TRUE,
            ) %>%
            drop_na(Phrase_Surprisal) %>% # Drops values w/out valid surprisal value
            group_by(Sentence_ID) %>% # Drops sentences without one subject and one object
                filter(n() == 2 & n_distinct(NP_Argument) == 2) %>%
                ungroup()

df_filtered <- df %>% 
        select(Sentence_ID, Sentence_Text, Phrase_Token, surprisal, definiteness, argPos, Word_Token_Index, within_file_id, within_chunk_id, np_start_idx)

saveRDS(df, file = "Results 12-9 5PM/filtered_12-9_5PM.rds")
write_csv(df, "Results 12-9 5PM/filtered_12-9_5PM.csv")